# Postposition Analysis: Hindi-HDTB (UD)

This notebook studies how Hindi **postpositions** (tokens with `deprel = case`) relate to the **dependency labels of their parent words**.

We are collecting evidence for later Karaka rule design. **No Karaka rules are implemented here.**

**Research question:** When a postposition appears, what syntactic role does the word it attaches to usually play?

**Example:** In *शाहजेहन **ने** बनवाया*, the postposition `ने` has `deprel = case` and attaches to `शाहजेहन`, whose label is `nsubj`.

## What We Record

For every token where `deprel = case`:

1. **Postposition** — the `form` of the case token (e.g. `ने`, `को`, `से`)
2. **Parent label** — the `deprel` of the token's head (the noun/pronoun the postposition attaches to)

We then count how often each postposition appears and which parent labels co-occur with it.

## 1. Load the CONLL-U File

Same loader as in `01_dataset_exploration.ipynb`.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path


def load_conllu(filepath):
    """Read a CONLL-U file and return a list of sentence dictionaries."""
    sentences = []
    current = {"text": "", "sent_id": "", "tokens": []}

    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line:
                if current["tokens"]:
                    sentences.append(current)
                    current = {"text": "", "sent_id": "", "tokens": []}
                continue

            if line.startswith("#"):
                if line.startswith("# text = "):
                    current["text"] = line[len("# text = "):]
                elif line.startswith("# sent_id = "):
                    current["sent_id"] = line[len("# sent_id = "):]
                continue

            columns = line.split("\t")
            if len(columns) < 8 or "-" in columns[0]:
                continue

            current["tokens"].append({
                "id": columns[0],
                "form": columns[1],
                "lemma": columns[2],
                "head": columns[6],
                "deprel": columns[7],
            })

    if current["tokens"]:
        sentences.append(current)

    return sentences


data_path = Path("../data/raw/hi_hdtb-ud-train.conllu")
sentences = load_conllu(data_path)

print(f"Loaded {len(sentences)} sentences from {data_path}")

Loaded 13306 sentences from ..\data\raw\hi_hdtb-ud-train.conllu


## 2. Extract Case Token Records

Walk through every sentence. When we find a `case` token, look up its head and store
(postposition, parent_deprel) as one observation.

In [2]:
def extract_case_records(sentences):
    """
    Return a list of (postposition, parent_deprel) tuples.

    Each tuple is one case token and the dependency label of its parent.
    """
    records = []

    for sentence in sentences:
        # Map token id -> token for quick head lookup
        id_to_token = {token["id"]: token for token in sentence["tokens"]}

        for token in sentence["tokens"]:
            if token["deprel"] != "case":
                continue

            parent = id_to_token.get(token["head"])
            if parent is None:
                continue

            records.append((token["form"], parent["deprel"]))

    return records


case_records = extract_case_records(sentences)

print(f"Total case tokens: {len(case_records)}")
print(f"Unique postpositions: {len(set(pp for pp, _ in case_records))}")

Total case tokens: 53121
Unique postpositions: 139


## 3. Helper: Print a Frequency Table

A small helper to display counts and percentages in a readable table.

In [3]:
def print_frequency_table(counter, title, total=None, top_n=None):
    """Print a ranked frequency table from a Counter."""
    if total is None:
        total = sum(counter.values())

    items = counter.most_common(top_n)

    print(title)
    print(f"{'Item':<16} {'Count':>8} {'Percent':>10}")
    print("-" * 36)

    for item, count in items:
        percent = 100 * count / total if total else 0
        print(f"{item:<16} {count:>8} {percent:>9.1f}%")

    print()

## 4. Most Common Postpositions

How often does each postposition appear as a `case` token in the training data?

In [4]:
postposition_counts = Counter(pp for pp, _ in case_records)

print_frequency_table(
    postposition_counts,
    title="Top 20 postpositions (deprel = case)",
    total=len(case_records),
    top_n=20,
)

Top 20 postpositions (deprel = case)
Item                Count    Percent
------------------------------------
के                  11001      20.7%
में                  8115      15.3%
को                   5799      10.9%
की                   5514      10.4%
ने                   4862       9.2%
से                   4324       8.1%
पर                   2956       5.6%
का                   2947       5.5%
लिए                  1080       2.0%
तक                    655       1.2%
साथ                   609       1.1%
बाद                   544       1.0%
द्वारा                392       0.7%
बीच                   310       0.6%
बारे                  296       0.6%
दौरान                 283       0.5%
मुताबिक               271       0.5%
खिलाफ                 226       0.4%
पहले                  214       0.4%
ओर                    203       0.4%



## 5. Parent Dependency Labels per Postposition

For each postposition, which **parent `deprel`** appears most often?

Below we show the top 5 parent labels for the **20 most frequent** postpositions.

In [5]:
# Group records by postposition
parent_labels_by_postposition = defaultdict(list)
for postposition, parent_deprel in case_records:
    parent_labels_by_postposition[postposition].append(parent_deprel)


def show_parent_label_distribution(postposition, top_n=5):
    """Print parent deprel counts for one postposition."""
    parent_deprels = parent_labels_by_postposition[postposition]
    counter = Counter(parent_deprels)
    total = len(parent_deprels)

    print(f"Postposition: {postposition}  (total: {total})")
    print(f"{'Parent label':<16} {'Count':>8} {'Percent':>10}")
    print("-" * 36)

    for label, count in counter.most_common(top_n):
        percent = 100 * count / total
        print(f"{label:<16} {count:>8} {percent:>9.1f}%")

    print()


# Show parent-label breakdown for the 20 most common postpositions
top_postpositions = [pp for pp, _ in postposition_counts.most_common(20)]

for postposition in top_postpositions:
    show_parent_label_distribution(postposition, top_n=5)

Postposition: के  (total: 11001)
Parent label        Count    Percent
------------------------------------
nmod                 6481      58.9%
obl                  4005      36.4%
conj                  253       2.3%
nsubj                 152       1.4%
obj                    42       0.4%

Postposition: में  (total: 8115)
Parent label        Count    Percent
------------------------------------
obl                  7108      87.6%
nmod                  769       9.5%
conj                  107       1.3%
root                   70       0.9%
obj                    36       0.4%

Postposition: को  (total: 5799)
Parent label        Count    Percent
------------------------------------
obj                  2624      45.2%
iobj                 1389      24.0%
obl                  1021      17.6%
nsubj                 560       9.7%
conj                  123       2.1%

Postposition: की  (total: 5514)
Parent label        Count    Percent
------------------------------------
nmod            

## 6. Focus Postpositions for Karaka Rule Design

These five postpositions are central to the project's initial hypotheses
(see `docs/ud_to_karaka_mapping_v1.md` and `docs/project_context.md`):

| Postposition | Initial hypothesis (not yet a rule) |
|--------------|-------------------------------------|
| `ने` | Often signals agent / Kartā context |
| `को` | Often signals Karma or Sampradāna |
| `से` | Often signals Karaṇa or Apādāna |
| `में` | Often signals Adhikaraṇa (location) |
| `पर` | Often signals Adhikaraṇa (location/surface) |

The tables below show what **parent UD labels** actually co-occur in Hindi-HDTB.

In [6]:
FOCUS_POSTPOSITIONS = ["ने", "को", "से", "में", "पर"]


def summarize_focus_postposition(postposition):
    """Print detailed statistics for one focus postposition."""
    if postposition not in parent_labels_by_postposition:
        print(f"Postposition not found in data: {postposition}")
        print()
        return

    parent_deprels = parent_labels_by_postposition[postposition]
    counter = Counter(parent_deprels)
    total = len(parent_deprels)
    share_of_all_case = 100 * total / len(case_records)

    print("=" * 60)
    print(f"Postposition: {postposition}")
    print(f"Total occurrences: {total}")
    print(f"Share of all case tokens: {share_of_all_case:.1f}%")
    print(f"Most common parent label: {counter.most_common(1)[0][0]} "
          f"({100 * counter.most_common(1)[0][1] / total:.1f}%)")
    print()

    print(f"{'Parent label':<16} {'Count':>8} {'Percent':>10}")
    print("-" * 36)
    for label, count in counter.most_common():
        percent = 100 * count / total
        print(f"{label:<16} {count:>8} {percent:>9.1f}%")
    print()


for postposition in FOCUS_POSTPOSITIONS:
    summarize_focus_postposition(postposition)

Postposition: ने
Total occurrences: 4862
Share of all case tokens: 9.2%
Most common parent label: nsubj (98.4%)

Parent label        Count    Percent
------------------------------------
nsubj                4785      98.4%
conj                   62       1.3%
nsubj:pass             10       0.2%
obl                     3       0.1%
obj                     1       0.0%
dislocated              1       0.0%

Postposition: को
Total occurrences: 5799
Share of all case tokens: 10.9%
Most common parent label: obj (45.2%)

Parent label        Count    Percent
------------------------------------
obj                  2624      45.2%
iobj                 1389      24.0%
obl                  1021      17.6%
nsubj                 560       9.7%
conj                  123       2.1%
nsubj:pass             39       0.7%
nmod                   35       0.6%
acl                     6       0.1%
xcomp                   1       0.0%
root                    1       0.0%

Postposition: से
Total occurrence

## 7. Summary Table: Focus Postpositions at a Glance

A compact comparison of the five focus postpositions — total count and top parent label.

In [7]:
print(f"{'Postposition':<12} {'Count':>8} {'Top parent':<14} {'Top %':>8}")
print("-" * 46)

for postposition in FOCUS_POSTPOSITIONS:
    parent_deprels = parent_labels_by_postposition[postposition]
    counter = Counter(parent_deprels)
    total = len(parent_deprels)
    top_label, top_count = counter.most_common(1)[0]
    top_percent = 100 * top_count / total

    print(
        f"{postposition:<12} {total:>8} {top_label:<14} {top_percent:>7.1f}%"
    )

Postposition    Count Top parent        Top %
----------------------------------------------
ने               4862 nsubj             98.4%
को               5799 obj               45.2%
से               4324 obl               69.5%
में              8115 obl               87.6%
पर               2956 obl               89.8%


## 8. Initial Observations (Evidence Only)

These are **data observations**, not Karaka rules:

- **`ने`** attaches almost exclusively to `nsubj` parents — strong evidence linking `ने` to subject/agent contexts.
- **`को`** appears with mixed parents: `obj`, `iobj`, and `obl` are all common — ambiguity expected for Karma vs Sampradāna rules.
- **`से`** most often attaches to `obl` parents, but `nmod`, `obj`, and `iobj` also appear.
- **`में`** and **`पर`** are dominated by `obl` parents — consistent with locative/adverbial uses.
- Genitive markers (`के`, `की`, `का`) are the most frequent case tokens overall; they link noun phrases rather than marking Karaka directly.

Record further notes in `docs/research_notes.md` after reviewing the printed tables.

## Next Steps

1. Document surprising or ambiguous patterns in `docs/research_notes.md`.
2. Refine `docs/ud_to_karaka_mapping_v1.md` where data contradicts initial hypotheses.
3. Design symbolic verification rules in `src/verifier/` based on high-confidence patterns (e.g. `ने` + `nsubj`).